### 1. 임베딩
- 텍스트를 문장 단위로 벡터 : [0.25, 0.12, 0.11]으로 변환 예시로 들 수 있다
- 검색, 추천시스템, RAG(챗봇) -> 벡터 변환형식

### 2. 임베딩 모델 준비 
- 한국어에 강한 최신 모델 : 경량화된 모델 사용 -> cpu 사용 가능

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("dragonkue/multilingual-e5-small-ko-v2")
print("모델 준비 완료")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\study-with-ai\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lee\.cache\huggingface\hub\models--dragonkue--multilingual-e5-small-ko-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/217 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/23.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/965 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

모델 준비 완료


### E5 - 모델 사용법 익히기 - 
- query : 질문 입력할 때
- passage : 검색 대상이 되는 문서

In [3]:
sentence = "오늘 점심 메뉴는?"
with_prefix=model.encode("query:" + sentence, normalize_embeddings=True) # 2번재 파라미터 유사도 계산 
without_prefix=model.encode(sentence, normalize_embeddings=True) # 2번재 파라미터 유사도 계산 

print("접두사가 있을 때", with_prefix[:10])
print("접두사가 있을 때", without_prefix[:10])

접두사가 있을 때 [ 0.03057274 -0.00737129 -0.09526572  0.01687642  0.07531504 -0.04751493
  0.03268289  0.01705527  0.04623772  0.063585  ]
접두사가 있을 때 [ 0.02251527 -0.00308811 -0.0859407   0.00292922  0.088636   -0.04761829
  0.04911163  0.01086638  0.05150406  0.07169256]


In [ ]:
# sentence = ["오늘 점심으로 무엇을 먹을까요?",
#             "오늘 점심은 일식인가요?",
#             "오늘 점심은 중식인가요?",
#             "오늘 점심은 저녁에 어떤 메뉴를 먹을 지 고민해봐요",
#             "저녁에 뭐해요?",
#             "주식은 언제 떡상 할까요?"]

sentence = ["오늘 점심으로 무엇을 먹을까요?",
             "오늘 점심은 일식인가요?",
             "오늘 점심은 중식인가요?",
             "오늘 점심은 저녁에 어떤 메뉴를 먹을 지 고민해봐요",
             "저녁에 뭐해요?",
             "주식은 언제 떡상 할까요?"]

vecs = model.encode(sentence, normalize_embeddings=True)

In [8]:
vecs.shape

(6, 384)

In [39]:
# 기사 내용을 통째로 벡터로 바꾸면
import pandas as pd

df = pd.read_csv("../data/11-1_뉴스정제.csv") 
df['정제본문'][0]


'서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복합몰을 만든다고 6일 밝혔다'

In [38]:
df['passage_add'] = df['정제본문'].apply(lambda x: "passage"+ x)
df.head(1)

,제목,본문,카테고리,요약,출처URL,정제본문,passage_add
0,현대백화점그룹 더현대 광주 추진,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...,경제,"6 6일 현대백화점그룹이 광주시에 문화복합몰을 만든다고 6일 밝혔으며, 광주시는 서...",https://n.news.naver.com/mnews/article/001/001...,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...,passage서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과...


In [41]:
article_vec=model.encode(df['정제본문'].tolist()[:100], normalize_embeddings=True)
print(article_vec)

[[ 0.04744077  0.04341251 -0.04399779 ...  0.06326543 -0.08641391
   0.01726929]
 [ 0.03891128  0.04919324 -0.08968784 ...  0.01063687 -0.07118414
   0.0072831 ]
 [ 0.0992108  -0.0204195   0.01587223 ...  0.08357667 -0.03688203
  -0.04291758]
 ...
 [ 0.01680643  0.01068763 -0.03125948 ...  0.01977171  0.02480885
   0.0051817 ]
 [ 0.04114025  0.01449313 -0.08008035 ...  0.02991807 -0.01783201
   0.03646773]
 [ 0.06273349 -0.05472916 -0.03805257 ...  0.03129239 -0.0331208
   0.00247542]]


In [44]:
from sentence_transformers import util

query = "경제 상황이 어떻게 되고 있어?"

def search_text(query, k=3):    # 질문과 유사한 문서 최대 k 개 가져오기
    query_vec = model.encode("query: " + query, normalize_embeddings=True)
    similarity = util.cos_sim(query_vec, article_vec)[0]
    for i in similarity.argsort(descending=True)[:k]:
        i = int(i)
        print(f"{similarity[i]}, {df['정제본문'].iloc[i]}")

In [45]:
search_text(query)

0.5671266317367554, 경제와이드 모닝벨 조간 브리핑 장연재 조간브리핑입니다 발길 끊는 개미들 거래대금 20개월만에 최저 동학개미들이 국내 주식시장을 떠나고 있다는 기사 먼저 보겠습니다 중앙일보입니다 미국발 금리 인상과 세계 경제 침체 우려로 올해에만 코스피가 20 넘게 급락하자 개인투자자들이 증시를 이탈하고 있습니다 코스피에서 개인 투자자의 하루 평균 거래 대금은 4조 3 009억 원으로 2년 4개월 만에 최저 수준을 기록했습니다 증시 대기자금인 투자자예탁금은 지난달 말 기준 반년 만에 10조 원이 줄었고 빚을 내 투자한 후 아직 갚지 않은 금액인 신용거래융자 잔고도 같은 기간 5조 원 이상 줄었습니다 이런 가운데 당분간 전 세계의 경기 침체 공포로 국내증시 반등을 기대하긴 어렵다는 의견이 지배적인데요 반면 일부 증권가에서는 기존 악재가 반영된 만큼 약세장을 오히려 기회로 삼아야 한다는 의견도 있습니다 지방 중심 역전세 깡통아파트 확산 경향신문 기사입니다 최근 지방의 저가 아파트를 중심으로 전세가격이 매매가격보다 높은 역전세 현상이 나타나고 있습니다 정부의 다주택자 규제 완화 조치 이후 매물이 쌓이고 있지만 금리가 오르자 집을 사려는 사람이 없어 집값은 하락하고 있습니다 반면 전세시장은 신규 계약체결 시점에 경신까지 염두에 두고 4년 치 상승분을 미리 가격에 반영하려는 경향이 있어서 역전세가 발생하고 있는 건데요 전세계약 시점에 집값이 이미 전세가격보다 낮다면 보증금을 보호받을 수 없어서 문제가 되고 있습니다 이런 현상은 현재까지 갭투자가 주로 이뤄지는 3억 원 이하 지방 저가 아파트에서 주로 나타나는데요 전문가들은 서울 집값은 큰 변화가 없기 때문에 전국적인 문제로 보기는 어렵다고 분석했습니다 경기 침체에 빅테크 감원 시작됐다 전 세계 기업에 감원 공포가 불고 있다는 조선일보 기사도 보겠습니다 코로나 팬데믹 기간 큰돈을 벌며 조직 규모를 키웠던 테크 기업들 너도나도 긴축 경영에 돌입하고 있습니다 인플레이션과 금리 인상 등으로 사업 환경이 급속도

In [46]:
query = "오늘 점심메뉴 뭐야?"
query = "요새 경제도 않좋은데 가격까지 고려한 오늘 점심 메뉴가 뭐야?"
search_text(query)

0.5128874182701111, 유류세 인하 폭이 37 로 확대 적용된 1일 오후 서울 서초구 서울만남의광장 휴게소 주유소에 주유를 위해 대기중인 차량이 줄지어 서 있다 유류세 인하로 이 주유소에서는 휘발유가 리터당 2068원 경유가 2152원으로 판매되고 있다 이날 오전 9시 기준 전국 평균 휘발유 가격은 전날보다 11 37원 내린 리터당 2133 53원을 나타냈다 전국 평균 경유 가격도 전날보다 7 38원 내린 2160 28원을 기록했다 사진영상기획부 발로 뛰는 더팩트는 24시간 여러분의 제보를 기다립니다
0.49696770310401917, 경제와이드 모닝벨 조간 브리핑 장연재 조간브리핑입니다 발길 끊는 개미들 거래대금 20개월만에 최저 동학개미들이 국내 주식시장을 떠나고 있다는 기사 먼저 보겠습니다 중앙일보입니다 미국발 금리 인상과 세계 경제 침체 우려로 올해에만 코스피가 20 넘게 급락하자 개인투자자들이 증시를 이탈하고 있습니다 코스피에서 개인 투자자의 하루 평균 거래 대금은 4조 3 009억 원으로 2년 4개월 만에 최저 수준을 기록했습니다 증시 대기자금인 투자자예탁금은 지난달 말 기준 반년 만에 10조 원이 줄었고 빚을 내 투자한 후 아직 갚지 않은 금액인 신용거래융자 잔고도 같은 기간 5조 원 이상 줄었습니다 이런 가운데 당분간 전 세계의 경기 침체 공포로 국내증시 반등을 기대하긴 어렵다는 의견이 지배적인데요 반면 일부 증권가에서는 기존 악재가 반영된 만큼 약세장을 오히려 기회로 삼아야 한다는 의견도 있습니다 지방 중심 역전세 깡통아파트 확산 경향신문 기사입니다 최근 지방의 저가 아파트를 중심으로 전세가격이 매매가격보다 높은 역전세 현상이 나타나고 있습니다 정부의 다주택자 규제 완화 조치 이후 매물이 쌓이고 있지만 금리가 오르자 집을 사려는 사람이 없어 집값은 하락하고 있습니다 반면 전세시장은 신규 계약체결 시점에 경신까지 염두에 두고 4년 치 상승분을 미리 가격에 반영하려는 경향이 있어서 역전세가 발생하고 있는 건데요 전세계약 시점에 집값이

In [49]:
query = "대한민국 악재 영향 찾아줘"
search_text(query,5)

0.5788738131523132, 한은 우크라 사태 국내 물가 영향 원자재 해외 의존도 높은 산업 타격 러시아의 우크라이나 침공 중국 봉쇄 조치 등 최근 글로벌 공급망 차질이 기업의 비용 부담으로 이어져 이미 천정부지로 치솟는 물가 오름세가 더 커질 수 있다는 경고가 나왔다 공급망 차질이 지속되면 자동차와 2차전지 등 일부 업종은 생산에도 영향을 받을 것으로 보인다 한국은행은 4일 발표한 최근 글로벌 공급망 차질의 특징 및 국내 산업에 미치는 영향 보고서에서 우크라이나 사태 장기화 글로벌 식량 수급 불안 중국의 제로 코로나 정책 유지 등으로 향후 글로벌 공급망의 불확실성이 크다 며 이런 리스크가 현실화하면 대외 의존도가 높은 우리나라는 물가 오름세가 심화하고 생산에 대한 영향도 확대될 가능성이 있다 고 진단했다 보고서에 따르면 글로벌 공급망 차질로 인해 자동차 건설 기계장비 등 일부 산업은 부품 자재 수급 차질이 빚어져 생산이 일부 제약됐다 직접적인 생산 차질은 다른 국가들과 비교해 상대적으로 큰 편은 아니라는 게 한은의 설명이다 아울러 원자재 중간재 가격이 상승하면서 대부분 산업에서 비용 부담은 커졌다 실제로 생산단계별 물가를 보면 5월 기준으로 원재료는 1년 전보다 60 8 상승했고 중간재는 15 4 나 뛰었다 생산자물가 통계에서 공산품으로 분류된 품목 중 가격 상승률이 5 이상인 품목의 비중은 절반을 넘었고 가격 상승률이 10 이상인 품목은 약 40 에 달한다 생산자들이 제품을 만드는 데 드는 비용이 늘어난 만큼 앞으로 소비자물가에도 영향을 미칠 전망이다 MobileAdNew center 한은은 우크라이나 사태 등 공급망 차질이 길어지면 원자재 해외 의존도가 높은 우리 산업에 부정적인 영향을 미칠 것으로 봤다 보고서는 자동차 2차전지 등에 사용되는 주요 필수 소재인 마그네슘 희토류 리튬 등의 대중 의존도는 80 이상이며 크립톤 제논 팔라듐 등의 대러시아 우크라이나 의존도는 30 이상 이라며 글로벌 공급망 상황과 국내 산업의 취약성을 면밀히 점검하고 향후

### 실습하기
- 내가 질문하려는 주제와 연관있는 데이터 생성
    - ex) 점심 메뉴 추천이라면
            - 리스트에 점심메뉴 관련 데이터를 데이터 프레임 형태로 만들어서 벡터화 시키자
- 질문을 걸어 얼마나 잘 찾는지 확인

In [51]:
## 1. 상품 데이터 만들기

import pandas as pd

product_df = pd.DataFrame({
    "product_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "product": [
        "경량 노트북",
        "게이밍 노트북",
        "무선 이어폰",
        "노이즈 캔슬링 헤드폰",
        "기계식 키보드",
        "인체공학 마우스",
        "휴대용 보조배터리",
        "USB-C 멀티 허브"
    ],
    "category": [
        "노트북", "노트북", "오디오", "오디오",
        "주변기기", "주변기기", "모바일", "주변기기"
    ],
    "description": [
        "가볍고 배터리가 오래가는 노트북으로 문서 작성과 인터넷 검색에 적합하다.",
        "고성능 그래픽카드가 탑재되어 게임과 영상 편집에 적합하다.",
        "작고 가벼운 무선 이어폰으로 출퇴근과 운동할 때 사용하기 좋다.",
        "외부 소음을 차단해 음악 감상과 장시간 집중에 적합한 헤드폰이다.",
        "타건감이 좋고 내구성이 뛰어나 개발자와 사무직 사용자에게 적합하다.",
        "손목 부담을 줄여주는 디자인으로 장시간 컴퓨터 작업에 적합하다.",
        "휴대하기 좋은 작은 크기로 여행이나 외출 시 스마트폰 충전에 유용하다.",
        "노트북에 연결해 HDMI, USB, SD카드 등을 동시에 사용할 수 있다."
    ],
    "price": [
        800000, 1800000, 120000, 350000,
        150000, 80000, 50000, 70000
    ]
})

In [52]:
## 2. 벡터화할 텍스트 만들기

# 상품명, 카테고리, 설명을 하나의 문장으로 합칩니다.

product_df["text"] = (
    "상품명: " + product_df["product"] +
    " 카테고리: " + product_df["category"] +
    " 설명: " + product_df["description"]
)

product_df[["product", "text"]].head()

,product,text
0,경량 노트북,상품명: 경량 노트북 카테고리: 노트북 설명: 가볍고 배터리가 오래가는 노트북으로 ...
1,게이밍 노트북,상품명: 게이밍 노트북 카테고리: 노트북 설명: 고성능 그래픽카드가 탑재되어 게임과...
2,무선 이어폰,상품명: 무선 이어폰 카테고리: 오디오 설명: 작고 가벼운 무선 이어폰으로 출퇴근과...
3,노이즈 캔슬링 헤드폰,상품명: 노이즈 캔슬링 헤드폰 카테고리: 오디오 설명: 외부 소음을 차단해 음악 감...
4,기계식 키보드,상품명: 기계식 키보드 카테고리: 주변기기 설명: 타건감이 좋고 내구성이 뛰어나 개...


In [53]:
## 3. 상품 벡터 만들기

passages = [
    "passage: " + text
    for text in product_df["text"].tolist()
]

product_vec = model.encode(
    passages,
    normalize_embeddings=True
)

print(product_vec.shape)

(8, 384)


In [54]:
## 4. 질문으로 상품 검색하기

from sentence_transformers import util

question = "가볍고 오래 사용할 수 있으며 문서 작업에 좋은 노트북"

query_vec = model.encode(
    "query: " + question,
    normalize_embeddings=True
)

scores = util.cos_sim(query_vec, product_vec)[0]

result_df = product_df.copy()
result_df["similarity"] = scores.cpu().numpy()

result_df = result_df.sort_values(
    "similarity",
    ascending=False
)

result_df[
    ["product", "category", "description", "price", "similarity"]
].head(3)

,product,category,description,price,similarity
0,경량 노트북,노트북,가볍고 배터리가 오래가는 노트북으로 문서 작성과 인터넷 검색에 적합하다.,800000,0.765813
1,게이밍 노트북,노트북,고성능 그래픽카드가 탑재되어 게임과 영상 편집에 적합하다.,1800000,0.638839
4,기계식 키보드,주변기기,타건감이 좋고 내구성이 뛰어나 개발자와 사무직 사용자에게 적합하다.,150000,0.586016


In [55]:
## 5. 여러 질문 테스트하기

questions = [
    "게임과 영상 편집을 할 수 있는 고성능 컴퓨터",
    "출퇴근할 때 사용할 작고 가벼운 음악 기기",
    "주변 소음을 차단하고 공부할 수 있는 제품",
    "장시간 컴퓨터를 사용해도 손목이 편한 제품",
    "여행 중 스마트폰을 충전할 수 있는 제품"
]

for question in questions:
    query_vec = model.encode(
        "query: " + question,
        normalize_embeddings=True
    )

    scores = util.cos_sim(query_vec, product_vec)[0]
    top_index = scores.argmax().item()

    print("질문:", question)
    print("추천 상품:", product_df.iloc[top_index]["product"])
    print()



질문: 게임과 영상 편집을 할 수 있는 고성능 컴퓨터
추천 상품: 게이밍 노트북

질문: 출퇴근할 때 사용할 작고 가벼운 음악 기기
추천 상품: 무선 이어폰

질문: 주변 소음을 차단하고 공부할 수 있는 제품
추천 상품: 노이즈 캔슬링 헤드폰

질문: 장시간 컴퓨터를 사용해도 손목이 편한 제품
추천 상품: 인체공학 마우스

질문: 여행 중 스마트폰을 충전할 수 있는 제품
추천 상품: 휴대용 보조배터리



In [57]:
## 6. 가격 조건까지 적용하기

#   벡터 검색은 의미가 비슷한 상품을 찾고, 가격 같은 조건은 별도로 필터링하는 것이 좋습니다.

question = "제일 많이 팔리는 상품"
max_price = 1_000_000

filtered_df = product_df[
    (product_df["category"] == "노트북") &
    (product_df["price"] <= max_price)
].copy()

filtered_vec = model.encode(
    ["passage: " + text for text in filtered_df["text"]],
    normalize_embeddings=True
)

query_vec = model.encode(
    "query: " + question,
    normalize_embeddings=True
)

scores = util.cos_sim(query_vec, filtered_vec)[0]
filtered_df["similarity"] = scores.cpu().numpy()

filtered_df.sort_values(
    "similarity",
    ascending=False
)[["product", "price", "description", "similarity"]]

,product,price,description,similarity
0,경량 노트북,800000,가볍고 배터리가 오래가는 노트북으로 문서 작성과 인터넷 검색에 적합하다.,0.443388


In [58]:
## 1. 라이브러리 불러오기
import pandas as pd
from sentence_transformers import SentenceTransformer, util

## 2. 모델 불러오기
model = SentenceTransformer(
    "intfloat/multilingual-e5-small",
    device="cpu"
)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

c:\study-with-ai\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lee\.cache\huggingface\hub\models--intfloat--multilingual-e5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [59]:
  ## 3. 의상 데이터 준비
clothes_df = pd.DataFrame({
      "상품명": [
          "오버핏 린넨 셔츠",
          "슬림핏 블랙 슬랙스",
          "베이직 화이트 티셔츠",
          "캐주얼 데님 재킷",
          "플리츠 롱스커트",
          "경량 패딩 점퍼"
      ],
      "카테고리": [
          "셔츠", "바지", "티셔츠",
          "재킷", "스커트", "아우터"
      ],
      "색상": [
          "아이보리", "검정", "흰색",
          "청색", "베이지", "검정"
      ],
      "소재": [
          "린넨", "폴리에스터", "면",
          "데님", "폴리에스터", "나일론"
      ],
      "스타일": [
          "캐주얼", "오피스", "베이직",
          "캐주얼", "여성스러운", "스포티"
      ],
      "계절": [
          "여름", "사계절", "사계절",
          "봄가을", "봄가을", "겨울"
      ],
      "설명": [
          "통기성이 좋고 가벼워 더운 날씨에 입기 좋은 셔츠",
          "단정한 디자인으로 출근할 때 활용하기 좋은 바지",
          "부드러운 면 소재의 기본 티셔츠",
          "청바지와 잘 어울리는 캐주얼한 데님 아우터",
          "움직임이 편하고 여성스러운 분위기를 연출하는 스커트",
          "가볍고 보온성이 좋아 겨울철 외출에 적합한 점퍼"
      ]
  })

In [60]:
## 4. 벡터화할 문장 만들기 
clothes_df["text"] = (
    "상품명: " + clothes_df["상품명"] +
    " 카테고리: " + clothes_df["카테고리"] +
    " 색상: " + clothes_df["색상"] +
    " 소재: " + clothes_df["소재"] +
    " 스타일: " + clothes_df["스타일"] +
    " 계절: " + clothes_df["계절"] +
    " 설명: " + clothes_df["설명"]
)

clothes_df[["상품명", "text"]].head(2)

,상품명,text
0,오버핏 린넨 셔츠,상품명: 오버핏 린넨 셔츠 카테고리: 셔츠 색상: 아이보리 소재: 린넨 스타일: 캐...
1,슬림핏 블랙 슬랙스,상품명: 슬림핏 블랙 슬랙스 카테고리: 바지 색상: 검정 소재: 폴리에스터 스타일:...


In [61]:
## 5. 상품 문장 벡터화

#   E5 모델에서는 상품이나 문서에 passage: 를 붙입니다.

passages = [
    "passage: " + text
    for text in clothes_df["text"].tolist()
]

clothes_vec = model.encode(
    passages,
    normalize_embeddings=True,
    batch_size=8,
    show_progress_bar=True
)

print(clothes_vec.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(6, 384)


In [62]:
## 6. 질문으로 의상 검색하기

#   질문에는 query: 를 붙입니다.

question = "여름에 입기 좋고 시원하면서 출근할 때도 어울리는 옷"

query_vec = model.encode(
    ["query: " + question],
    normalize_embeddings=True
)

scores = util.cos_sim(query_vec, clothes_vec)[0]

result_df = clothes_df.copy()
result_df["similarity"] = scores.cpu().numpy()

result_df = result_df.sort_values(
    "similarity",
    ascending=False
)

result_df[
    ["상품명", "카테고리", "색상", "스타일", "similarity"]
].head(3)


,상품명,카테고리,색상,스타일,similarity
1,슬림핏 블랙 슬랙스,바지,검정,오피스,0.869822
0,오버핏 린넨 셔츠,셔츠,아이보리,캐주얼,0.868785
3,캐주얼 데님 재킷,재킷,청색,캐주얼,0.846643


In [63]:
## 7. 재사용 가능한 검색 함수 만들기

def recommend_clothes(question, top_k=3):
    query_vec = model.encode(
        ["query: " + question],
        normalize_embeddings=True
    )

    scores = util.cos_sim(query_vec, clothes_vec)[0]

    result = clothes_df.copy()
    result["similarity"] = scores.cpu().numpy()

    return (
        result
        .sort_values("similarity", ascending=False)
        .head(top_k)
    )

# 사용 방법:

recommend_clothes(
    "편하게 입을 수 있고 청바지와 잘 어울리는 옷",
    top_k=3
)

,상품명,카테고리,색상,소재,스타일,계절,설명,text,similarity
3,캐주얼 데님 재킷,재킷,청색,데님,캐주얼,봄가을,청바지와 잘 어울리는 캐주얼한 데님 아우터,상품명: 캐주얼 데님 재킷 카테고리: 재킷 색상: 청색 소재: 데님 스타일: 캐주얼...,0.866738
1,슬림핏 블랙 슬랙스,바지,검정,폴리에스터,오피스,사계절,단정한 디자인으로 출근할 때 활용하기 좋은 바지,상품명: 슬림핏 블랙 슬랙스 카테고리: 바지 색상: 검정 소재: 폴리에스터 스타일:...,0.843371
0,오버핏 린넨 셔츠,셔츠,아이보리,린넨,캐주얼,여름,통기성이 좋고 가벼워 더운 날씨에 입기 좋은 셔츠,상품명: 오버핏 린넨 셔츠 카테고리: 셔츠 색상: 아이보리 소재: 린넨 스타일: 캐...,0.843364


##### 1.처음 한 번만 실행합니다.
- %pip install datasets

In [ ]:
## 2. 데이터 불러오기

from datasets import load_dataset
import pandas as pd

dataset = load_dataset(
    "alexkstern/myntra_fashion_dataset",
    split="train"
)

fashion_df = dataset.to_pandas()

fashion_df.head()

## 3. 데이터 구조 확인

fashion_df.shape
fashion_df.columns
fashion_df.info()
fashion_df.isna().sum()

# 주요 컬럼은 다음과 같습니다.

# name
# sku
# mpn
# price
# in_stock
# currency
# brand
# description
# images
# gender

# images 컬럼은 이번 실습에서 사용하지 않습니다.

# ## 4. 의류 상품만 선택하기

# 이 데이터에는 가방, 향수, 생활용품도 일부 포함되어 있으므로 의류 관련 상품만 간단히 필터링합니다.

clothing_keywords = [
    "shirt", "t-shirt", "jeans", "trouser",
    "dress", "skirt", "jacket", "kurta",
    "shorts", "suit", "top", "saree",
    "tights", "clothing", "blazer"
]

pattern = "|".join(clothing_keywords)

clothes_df = fashion_df[
    fashion_df["name"]
    .fillna("")
    .str.lower()
    .str.contains(pattern, na=False)
].copy()

clothes_df.shape

# 필터링 결과를 확인합니다.

clothes_df[
    ["name", "brand", "price", "gender"]
].head(10)

## 5. 임베딩용 텍스트 만들기

# 상품명, 브랜드, 성별, 설명을 하나의 문장으로 합칩니다.

clothes_df["text"] = (
    "상품명: " + clothes_df["name"].fillna("").astype(str) +
    " 브랜드: " + clothes_df["brand"].fillna("").astype(str) +
    " 성별: " + clothes_df["gender"].fillna("").astype(str) +
    " 설명: " + clothes_df["description"].fillna("").astype(str)
)

## 6. 모델 불러오기

from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer(
    "intfloat/multilingual-e5-small",
    device="cpu"
)



<class 'pandas.DataFrame'>
RangeIndex: 308 entries, 0 to 307
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   name         308 non-null    str  
 1   sku          308 non-null    str  
 2   mpn          308 non-null    str  
 3   price        308 non-null    str  
 4   in_stock     308 non-null    bool 
 5   currency     308 non-null    str  
 6   brand        308 non-null    str  
 7   description  308 non-null    str  
 8   images       308 non-null    str  
 9   gender       308 non-null    str  
dtypes: bool(1), str(9)
memory usage: 293.1 KB


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

(156, 384)


In [79]:
## 7. 상품 벡터 생성

passages = [
"passage: " + text
for text in clothes_df["text"].tolist()
]

clothes_vec = model.encode(
passages,
normalize_embeddings=True,
batch_size=8,
show_progress_bar=True
)

print(clothes_vec.shape)



Batches:   0%|          | 0/20 [00:00<?, ?it/s]

(156, 384)


In [80]:
## 8. 질문으로 상품 검색하기

question = "여름에 입기 좋은 여성 캐주얼 셔츠"

query_vec = model.encode(
    ["query: " + question],
    normalize_embeddings=True
)

scores = util.cos_sim(query_vec, clothes_vec)[0]

result_df = clothes_df.copy()
result_df["similarity"] = scores.cpu().numpy()

result_df.sort_values(
    "similarity",
    ascending=False
)[["name", "brand", "gender", "price", "similarity"]].head(5)

## 9. 재사용 가능한 추천 함수

def recommend_clothes(question, top_k=5):
    query_vec = model.encode(
    ["query: " + question],
    normalize_embeddings=True
)

    scores = util.cos_sim(query_vec, clothes_vec)[0]

    result = clothes_df.copy()
    result["similarity"] = scores.cpu().numpy()

    return(
        result
        .sort_values("similarity", ascending=False)
        .head(top_k)
        )

In [86]:
from IPython.display import display, HTML

result = recommend_clothes(
    "가을에 입기 좋은 남성 화려한 패션",
    top_k=5
).copy()

# 여러 이미지 URL 중 첫 번째 URL만 사용
result["상품 링크"] = (
    result["images"]
    .fillna("")
    .str.split(" ~ ")
    .str[0]
    .apply(
        lambda url: (
            f'<a href="{url}" target="_blank">상품 보기</a>'
            if url else ""
        )
    )
)

result_korean = result.rename(columns={
    "name": "상품명",
    "brand": "브랜드",
    "gender": "성별",
    "price": "가격",
    "similarity": "유사도"
})

display(HTML(
    result_korean[
        ["상품명", "브랜드", "성별", "가격", "유사도", "상품 링크"]
    ].to_html(
        escape=False,
        index=False
    )
))

상품명,브랜드,성별,가격,유사도,상품 링크
HIGHLANDER Men White & Grey Slim Fit Striped Casual Shirt,HIGHLANDER,Men,699,0.858827,상품 보기
HIGHLANDER Men Purple & Beige Slim Fit Printed Casual Shirt,HIGHLANDER,Men,699,0.851430,상품 보기
HIGHLANDER Men Navy Blue & Maroon Slim Fit Checked Casual Shirt,HIGHLANDER,Men,699,0.847458,상품 보기
Parx Men Grey Slim Fit Printed Casual Shirt,Parx,Men,759,0.842807,상품 보기
HIGHLANDER Men Mustard & Black Slim Fit Checked Casual Shirt,HIGHLANDER,Men,699,0.842377,상품 보기


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\study-with-ai\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lee\.cache\huggingface\hub\models--dragonkue--BGE-m3-ko. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/31.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [95]:
load_dotenv()

True

1536